In [ ]:
common_kept = (
    set(kept_idx_lr)
    & set(kept_idx_rf)
    & set(kept_idx_gbr)
)

df_filtered = df.loc[list(common_kept)].copy()

print(f"Amostras mantidas (erro ≤ 25% nos 3 modelos): {len(df_filtered)}")
display(df_filtered)

Amostras mantidas (erro ≤ 25% nos 3 modelos): 205


,Tipo do concreto,fck [MPa],Unnamed: 4,l [mm],d [mm],l/d,Teor de fibra (%),N (ganchos),"fR,1 (N/mm²) (experimental)","fR,3 (N/mm²) (experimental)"
0,"AM-0,4-EH1",61.30000,7.82943,35,0.54997,63.64000,0.40000,1.00000,4.99000,3.32000
1,"AM-0,8-EH1",63.80000,7.98749,35,0.54997,63.64000,0.80000,1.00000,7.44000,5.68000
2,"AM-0,4-EH2",63.60000,7.97496,60,0.89996,66.67000,0.40000,1.00000,5.35000,5.53000
3,"AM-0,8-EH2",58.70000,7.66159,60,0.89996,66.67000,0.80000,1.00000,7.94000,8.78000
4,"AM-0,4-EH1",61.30000,7.82943,35,0.54997,63.64000,0.40000,1.00000,5.49000,3.03000
5,"AM-0,4-EH1",61.30000,7.82943,35,0.54997,63.64000,0.40000,1.00000,4.37000,3.28000
6,"AM-0,4-EH1",61.30000,7.82943,35,0.54997,63.64000,0.40000,1.00000,4.67000,2.98000
7,"AM-0,4-EH1",61.30000,7.82943,35,0.54997,63.64000,0.40000,1.00000,5.22000,3.06000
8,"AM-0,4-EH1",61.30000,7.82943,35,0.54997,63.64000,0.40000,1.00000,4.74000,3.50000
9,"AM-0,4-EH1",61.30000,7.82943,35,0.54997,63.64000,0.40000,1.00000,5.46000,4.09000


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_model(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²": r2_score(y_true, y_pred)
    }

In [ ]:
from sklearn.linear_model import LinearRegression

def run_linear_regression(df_data, label):
    # Usar os nomes exatos das colunas presentes no DataFrame
    X = df_data[[
        'fck             [MPa]',
        'Teor de fibra (%)'
    ]]
    y1 = df_data['fR,1 (N/mm²)      (experimental)']
    y3 = df_data['fR,3 (N/mm²)      (experimental)']

    lin_fr1 = LinearRegression().fit(X, y1)
    lin_fr3 = LinearRegression().fit(X, y3)

    res = []

    for name, model, y in [
        (f"Linear {label} — fR,1", lin_fr1, y1),
        (f"Linear {label} — fR,3", lin_fr3, y3)
    ]:
        y_pred = model.predict(X)
        metrics = evaluate_model(y, y_pred)
        metrics["Modelo"] = name
        metrics["Equação"] = (
            f"fR = {model.intercept_:.4f} "
            f"+ {model.coef_[0]:.4f}·fck "
            f"+ {model.coef_[1]:.4f}·fibra"
        )
        res.append(metrics)

    return pd.DataFrame(res)

In [ ]:
lr_full = run_linear_regression(df, "Amostra Completa")
lr_filt = run_linear_regression(df_filtered, "Erro ≤ 25%")

display(pd.concat([lr_full, lr_filt]))

,MAE,MSE,RMSE,R²,Modelo,Equação
0,0.96745,1.60926,1.26857,0.75129,"Linear Amostra Completa — fR,1",fR = -2.1999 + 0.0779·fck + 6.6752·fibra
1,1.45531,3.30189,1.81711,0.57662,"Linear Amostra Completa — fR,3",fR = -1.7317 + 0.0679·fck + 6.5830·fibra
0,0.70363,0.80804,0.89891,0.85646,"Linear Erro ≤ 25% — fR,1",fR = -2.0445 + 0.0838·fck + 6.3542·fibra
1,1.37204,2.87340,1.69511,0.61352,"Linear Erro ≤ 25% — fR,3",fR = -1.5086 + 0.0713·fck + 6.4330·fibra


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.ensemble import RandomForestRegressor

def train_dnn(X, y):
    model = Sequential([
        Dense(32, activation='relu', input_shape=(X.shape[1],)),
        Dense(16, activation='relu'),
        Dense(1)
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse'
    )

    model.fit(X, y, epochs=200, batch_size=16, verbose=0)
    return model

def run_random_forest(df_data, label):
    X = df_data[[
        'fck             [MPa]',
        'Teor de fibra (%)'
    ]].values
    y1 = df_data['fR,1 (N/mm²)      (experimental)']
    y3 = df_data['fR,3 (N/mm²)      (experimental)']

    rf_fr1 = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, y1)
    rf_fr3 = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, y3)

    res = []

    for name, model, y in [
        (f"Random Forest {label} — fR,1", rf_fr1, y1),
        (f"Random Forest {label} — fR,3", rf_fr3, y3)
    ]:
        y_pred = model.predict(X)
        metrics = evaluate_model(y, y_pred)
        metrics["Modelo"] = name
        metrics["Equação"] = "Random Forest (modelo caixa-preta)"
        res.append(metrics)

    return pd.DataFrame(res)

In [ ]:
def run_dnn(df_data, label):
    X = df_data[[
        'fck             [MPa]',
        'Teor de fibra (%)'
    ]].values
    y1 = df_data['fR,1 (N/mm²)      (experimental)']
    y3 = df_data['fR,3 (N/mm²)      (experimental)']

    dnn_fr1 = train_dnn(X, y1)
    dnn_fr3 = train_dnn(X, y3)

    res = []

    for name, model, y in [
        (f"DNN {label} — fR,1", dnn_fr1, y1),
        (f"DNN {label} — fR,3", dnn_fr3, y3)
    ]:
        y_pred = model.predict(X).ravel()
        metrics = evaluate_model(y, y_pred)
        metrics["Modelo"] = name
        metrics["Equação"] = "Rede Neural (modelo caixa-preta)"
        res.append(metrics)

    return pd.DataFrame(res)

In [ ]:
dnn_full = run_dnn(df, "Amostra Completa")
dnn_filt = run_dnn(df_filtered, "Erro ≤ 25%")

rf_full = run_random_forest(df, "Amostra Completa")
rf_filt = run_random_forest(df_filtered, "Erro ≤ 25%")

display(pd.concat([dnn_full, dnn_filt, rf_full, rf_filt]))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


,MAE,MSE,RMSE,R²,Modelo,Equação
0,0.92168,1.48699,1.21942,0.77018,"DNN Amostra Completa — fR,1",Rede Neural (modelo caixa-preta)
1,1.41823,3.06115,1.74962,0.60749,"DNN Amostra Completa — fR,3",Rede Neural (modelo caixa-preta)
0,0.81893,0.99313,0.99656,0.82358,"DNN Erro ≤ 25% — fR,1",Rede Neural (modelo caixa-preta)
1,1.38446,2.77294,1.66522,0.62703,"DNN Erro ≤ 25% — fR,3",Rede Neural (modelo caixa-preta)
0,0.32183,0.21681,0.46563,0.96649,"Random Forest Amostra Completa — fR,1",Random Forest (modelo caixa-preta)
1,0.46165,0.43451,0.65917,0.94429,"Random Forest Amostra Completa — fR,3",Random Forest (modelo caixa-preta)
0,0.24681,0.12227,0.34968,0.97828,"Random Forest Erro ≤ 25% — fR,1",Random Forest (modelo caixa-preta)
1,0.41857,0.36065,0.60054,0.95149,"Random Forest Erro ≤ 25% — fR,3",Random Forest (modelo caixa-preta)


In [ ]:
final_results = pd.concat([
    lr_full, lr_filt,
    rf_full, rf_filt,
    dnn_full, dnn_filt
]).reset_index(drop=True)

display(final_results)

,MAE,MSE,RMSE,R²,Modelo,Equação
0,0.96745,1.60926,1.26857,0.75129,"Linear Amostra Completa — fR,1",fR = -2.1999 + 0.0779·fck + 6.6752·fibra
1,1.45531,3.30189,1.81711,0.57662,"Linear Amostra Completa — fR,3",fR = -1.7317 + 0.0679·fck + 6.5830·fibra
2,0.70363,0.80804,0.89891,0.85646,"Linear Erro ≤ 25% — fR,1",fR = -2.0445 + 0.0838·fck + 6.3542·fibra
3,1.37204,2.87340,1.69511,0.61352,"Linear Erro ≤ 25% — fR,3",fR = -1.5086 + 0.0713·fck + 6.4330·fibra
4,0.32183,0.21681,0.46563,0.96649,"Random Forest Amostra Completa — fR,1",Random Forest (modelo caixa-preta)
5,0.46165,0.43451,0.65917,0.94429,"Random Forest Amostra Completa — fR,3",Random Forest (modelo caixa-preta)
6,0.24681,0.12227,0.34968,0.97828,"Random Forest Erro ≤ 25% — fR,1",Random Forest (modelo caixa-preta)
7,0.41857,0.36065,0.60054,0.95149,"Random Forest Erro ≤ 25% — fR,3",Random Forest (modelo caixa-preta)
8,0.92168,1.48699,1.21942,0.77018,"DNN Amostra Completa — fR,1",Rede Neural (modelo caixa-preta)
9,1.41823,3.06115,1.74962,0.60749,"DNN Amostra Completa — fR,3",Rede Neural (modelo caixa-preta)
